In [5]:
from pathlib import Path
import json
import pandas as pd

DATA_DIR = Path("../data/raw")

stations = [
    "toronto",
    "ottawa",
    "windsor",
    "sudbury",
    "thunder_bay"
]

weather_data = {}

for station in stations:
    file_path = DATA_DIR / f"weather_{station}_2025.json"

    with open(file_path, "r", encoding="utf-8") as file:
        data = json.load(file)

    df = pd.DataFrame(
        feature["properties"]
        for feature in data["features"]
    )

    df["timestamp"] = pd.to_datetime(df["LOCAL_DATE"])
    df["temperature"] = pd.to_numeric(
        df["TEMP"],
        errors="coerce"
    )

    df = (
        df[["timestamp", "temperature"]]
        .sort_values("timestamp")
        .reset_index(drop=True)
    )

    weather_data[station] = df

In [2]:
summary = []

for station, df in weather_data.items():
    summary.append({
        "station": station,
        "rows": len(df),
        "missing_temperature": df["temperature"].isna().sum(),
        "duplicates": df["timestamp"].duplicated().sum(),
        "start": df["timestamp"].min(),
        "end": df["timestamp"].max()
    })

summary_df = pd.DataFrame(summary)
summary_df

,station,rows,missing_temperature,duplicates,start,end
0,toronto,8760,1,0,2025-01-01,2025-12-31 23:00:00
1,ottawa,8758,1,0,2025-01-01,2025-12-31 23:00:00
2,windsor,8754,0,0,2025-01-01,2025-12-31 23:00:00
3,sudbury,8721,1,0,2025-01-01,2025-12-31 23:00:00
4,thunder_bay,7716,6,0,2025-01-01,2025-11-18 14:00:00


In [3]:
expected_hours = pd.date_range(
    "2025-01-01 00:00:00",
    "2025-12-31 23:00:00",
    freq="h"
)

for station, df in weather_data.items():
    missing_hours = expected_hours.difference(df["timestamp"])
    print(f"{station}: {len(missing_hours)} missing hours")
    print(missing_hours[:10])

toronto: 0 missing hours
DatetimeIndex([], dtype='datetime64[us]', freq='h')
ottawa: 2 missing hours
DatetimeIndex(['2025-07-03 07:00:00', '2025-07-03 09:00:00'], dtype='datetime64[us]', freq=None)
windsor: 6 missing hours
DatetimeIndex(['2025-08-05 10:00:00', '2025-08-05 11:00:00',
               '2025-08-05 12:00:00', '2025-08-05 13:00:00',
               '2025-08-05 14:00:00', '2025-08-05 15:00:00'],
              dtype='datetime64[us]', freq='h')
sudbury: 39 missing hours
DatetimeIndex(['2025-01-09 00:00:00', '2025-01-09 01:00:00',
               '2025-01-09 02:00:00', '2025-01-09 03:00:00',
               '2025-01-09 04:00:00', '2025-01-09 05:00:00',
               '2025-01-21 10:00:00', '2025-01-26 02:00:00',
               '2025-02-11 04:00:00', '2025-02-12 03:00:00'],
              dtype='datetime64[us]', freq=None)
thunder_bay: 1044 missing hours
DatetimeIndex(['2025-07-01 10:00:00', '2025-09-25 12:00:00',
               '2025-09-25 13:00:00', '2025-11-18 15:00:00',
          

In [4]:
thunder_bay_missing = expected_hours.difference(
    weather_data["thunder_bay"]["timestamp"]
)

print("First missing:", thunder_bay_missing.min())
print("Last missing:", thunder_bay_missing.max())

First missing: 2025-07-01 10:00:00
Last missing: 2025-12-31 23:00:00


In [6]:
expected_hours = pd.date_range(
    "2025-01-01 00:00:00",
    "2025-12-31 23:00:00",
    freq="h"
)

gap_summary = []

for station, df in weather_data.items():
    full = (
        df.set_index("timestamp")
        .reindex(expected_hours)
    )

    missing = full["temperature"].isna()

    # Group consecutive missing/non-missing periods
    groups = missing.ne(missing.shift()).cumsum()

    missing_gap_lengths = (
        missing.groupby(groups)
        .sum()
    )

    longest_gap = int(missing_gap_lengths.max())

    gap_summary.append({
        "station": station,
        "total_missing_temperature_hours": int(missing.sum()),
        "longest_consecutive_gap": longest_gap
    })

gap_summary_df = pd.DataFrame(gap_summary)

gap_summary_df

,station,total_missing_temperature_hours,longest_consecutive_gap
0,toronto,1,1
1,ottawa,3,1
2,windsor,6,6
3,sudbury,40,6
4,thunder_bay,11,5


In [7]:
for station, df in weather_data.items():
    full = (
        df.set_index("timestamp")
        .reindex(expected_hours)
    )

    missing_times = full.index[
        full["temperature"].isna()
    ]

    print(f"\n{station}")
    print("Missing temperature hours:", len(missing_times))
    print(missing_times[:20])


toronto
Missing temperature hours: 1
DatetimeIndex(['2025-08-05 12:00:00'], dtype='datetime64[us]', freq='h')

ottawa
Missing temperature hours: 3
DatetimeIndex(['2025-07-03 07:00:00', '2025-07-03 09:00:00',
               '2025-08-05 12:00:00'],
              dtype='datetime64[us]', freq=None)

windsor
Missing temperature hours: 6
DatetimeIndex(['2025-08-05 10:00:00', '2025-08-05 11:00:00',
               '2025-08-05 12:00:00', '2025-08-05 13:00:00',
               '2025-08-05 14:00:00', '2025-08-05 15:00:00'],
              dtype='datetime64[us]', freq='h')

sudbury
Missing temperature hours: 40
DatetimeIndex(['2025-01-09 00:00:00', '2025-01-09 01:00:00',
               '2025-01-09 02:00:00', '2025-01-09 03:00:00',
               '2025-01-09 04:00:00', '2025-01-09 05:00:00',
               '2025-01-21 10:00:00', '2025-01-26 02:00:00',
               '2025-02-11 04:00:00', '2025-02-12 03:00:00',
               '2025-02-21 00:00:00', '2025-02-23 21:00:00',
               '2025-02-23 2